In [17]:
import stim
import numpy as np
import itertools
import pymatching

def generate_3_round_data_only_circuit(flip_indices, model_prob=0.999):
    """
    Generates a full 3-round surface code circuit (Round 0, 1, 2) + Final Readout.
    Restricts active fault injection STRICTLY to data qubits during the Round 1 segment.
    """
    circuit = stim.Circuit()

    # --- QUBIT REGISTERS MAPPING ---
    def cq(i): return i
    def zq(i): return 9 + i
    def xq(i): return 13 + i
    all_xq = [xq(0), xq(1), xq(2), xq(3)]

    # Structural helper for the uniform syndrome extraction timeline
    def apply_syndrome_cycle(c):
        c.append("H", all_xq)
        
        # CNOT Layers 1-4
        c.append("CX", [cq(1), zq(1),  cq(3), zq(2),  cq(5), zq(3)])
        c.append("CX", [xq(1), cq(0),  xq(2), cq(4),  xq(3), cq(6)])
        c.append("CX", [cq(4), zq(1),  cq(6), zq(2),  cq(8), zq(3)])
        c.append("CX", [xq(1), cq(1),  xq(2), cq(5),  xq(3), cq(7)])
        
        # CNOT Layers 5-8
        c.append("CX", [cq(0), zq(0),  cq(2), zq(1),  cq(4), zq(2)])
        c.append("CX", [xq(0), cq(1),  xq(1), cq(3),  xq(2), cq(7)])
        c.append("CX", [cq(3), zq(0),  cq(5), zq(1),  cq(7), zq(2)])
        c.append("CX", [xq(0), cq(2),  xq(1), cq(4),  xq(2), cq(8)])
        c.append("H", all_xq)

    meas_targets = [xq(0), zq(0), xq(1), zq(1), xq(2), zq(2), xq(3), zq(3)]
    
    # ----------------------------------------------------
    # ROUND 0: Stable Baseline Initialization
    # ----------------------------------------------------
    apply_syndrome_cycle(circuit)
    circuit.append("M", meas_targets)
    circuit.append("R", meas_targets)
    
    # Initial deterministic Z-basis anchors
    circuit.append("DETECTOR", stim.target_rec(-7))  # zq0
    circuit.append("DETECTOR", stim.target_rec(-5))  # zq1
    circuit.append("DETECTOR", stim.target_rec(-3))  # zq2
    circuit.append("DETECTOR", stim.target_rec(-1))  # zq3

    # ----------------------------------------------------
    # ROUND 1: DATA FAULTS INJECTED HERE
    # ----------------------------------------------------
    if flip_indices:
        circuit.append("X_ERROR", [cq(i) for i in flip_indices], model_prob)
        
    apply_syndrome_cycle(circuit)
    circuit.append("M", meas_targets)
    circuit.append("R", meas_targets)
    
    # Natively tracks ancilla fluctuations relative to Round 0
    for i in range(8):
        circuit.append("DETECTOR", [stim.target_rec(-8 + i), stim.target_rec(-16 + i)])

    # ----------------------------------------------------
    # ROUND 2: Verification Buffer Round
    # ----------------------------------------------------
    apply_syndrome_cycle(circuit)
    circuit.append("M", meas_targets)
    
    # Tracks stabilization relative to Round 1
    for i in range(8):
        circuit.append("DETECTOR", [stim.target_rec(-8 + i), stim.target_rec(-16 + i)])

    # ----------------------------------------------------
    # FINAL READOUT DATA DESTRUCTION BOUNDARY
    # ----------------------------------------------------
    data_targets = [cq(i) for i in range(9)]
    circuit.append("M", data_targets)

    # Closes out the final stabilizer tracking frame (Data vs Round 2 Z-ancillas)
    circuit.append("DETECTOR", [stim.target_rec(-9), stim.target_rec(-6), stim.target_rec(-16)])  # zq0
    circuit.append("DETECTOR", [stim.target_rec(-8), stim.target_rec(-7), stim.target_rec(-5), stim.target_rec(-4), stim.target_rec(-14)])  # zq1
    circuit.append("DETECTOR", [stim.target_rec(-6), stim.target_rec(-5), stim.target_rec(-3), stim.target_rec(-2), stim.target_rec(-12)])  # zq2
    circuit.append("DETECTOR", [stim.target_rec(-4), stim.target_rec(-1), stim.target_rec(-10)])  # zq3

    circuit.append("OBSERVABLE_INCLUDE", [stim.target_rec(-9), stim.target_rec(-8), stim.target_rec(-7)], 0)

    return circuit

# --- MAIN EXECUTION BLOCK ---
if __name__ == "__main__":
    print("--- 3-Round Data-Only Dimensional Scaling Analysis via Safe X_ERROR ---")
    
    num_data_qubits = 9
    
    # STEP 1: Initialize graph topology using independent 1% error rates on data qubits
    base_circuit = generate_3_round_data_only_circuit(flip_indices=list(range(num_data_qubits)), model_prob=0.01)
    error_model = base_circuit.detector_error_model(decompose_errors=True)
    matching = pymatching.Matching.from_detector_error_model(error_model)
    
    # STEP 2: Loop through each dimensional weight layer
    for weight in range(1, num_data_qubits + 1):
        combinations = list(itertools.combinations(range(num_data_qubits), weight))
        decoder_failures = 0
        
        for flip_set in combinations:
            # Generate test circuit using 0.999 for stable execution decoding
            test_circuit = generate_3_round_data_only_circuit(flip_indices=flip_set, model_prob=0.999)
            
            sampler = test_circuit.compile_detector_sampler()
            det_samples, obs_samples = sampler.sample(shots=1, separate_observables=True)
            
            det_flips = det_samples[0]
            actual_obs = obs_samples[0]
            
            # Decode symptoms using uniform graph
            predicted_obs = matching.decode(det_flips)
            
            # Log instance if decoder fails to correct the combination
            if actual_obs[0] != predicted_obs[0]:
                decoder_failures += 1
                
        print(f"Dimension {weight} summary: {decoder_failures} / {len(combinations)} combinations defeated the decoder.")


--- 3-Round Data-Only Dimensional Scaling Analysis via Safe X_ERROR ---
Dimension 1 summary: 0 / 9 combinations defeated the decoder.
Dimension 2 summary: 18 / 36 combinations defeated the decoder.
Dimension 3 summary: 56 / 84 combinations defeated the decoder.
Dimension 4 summary: 57 / 126 combinations defeated the decoder.
Dimension 5 summary: 69 / 126 combinations defeated the decoder.
Dimension 6 summary: 28 / 84 combinations defeated the decoder.
Dimension 7 summary: 18 / 36 combinations defeated the decoder.
Dimension 8 summary: 9 / 9 combinations defeated the decoder.
Dimension 9 summary: 1 / 1 combinations defeated the decoder.


In [19]:
import stim
import numpy as np
import itertools
import pymatching

def generate_3_round_fully_loaded_circuit(data_flips, z_ancilla_flips, x_ancilla_flips, model_prob=0.999):
    """
    Generates a full 3-round surface code circuit (Round 0, 1, 2) + Final Readout.
    Restricts active fault injection STRICTLY to the Round 1 segment.
    """
    circuit = stim.Circuit()

    def cq(i): return i
    def zq(i): return 9 + i
    def xq(i): return 13 + i
    all_xq = [xq(0), xq(1), xq(2), xq(3)]

    # Structural helper for uniform syndrome extraction timeline
    def apply_syndrome_cycle(c, inject_errors=False):
        c.append("H", all_xq)
        
        # Inject X-ancilla faults right after H when they become active controls
        if inject_errors and x_ancilla_flips:
            c.append("X_ERROR", [xq(i) for i in x_ancilla_flips], model_prob)

        # CNOT Layers 1-4
        c.append("CX", [cq(1), zq(1),  cq(3), zq(2),  cq(5), zq(3)])
        c.append("CX", [xq(1), cq(0),  xq(2), cq(4),  xq(3), cq(6)])
        c.append("CX", [cq(4), zq(1),  cq(6), zq(2),  cq(8), zq(3)])
        c.append("CX", [xq(1), cq(1),  xq(2), cq(5),  xq(3), cq(7)])
        
        # Inject Z-ancilla faults mid-cycle
        if inject_errors and z_ancilla_flips:
            c.append("X_ERROR", [zq(i) for i in z_ancilla_flips], model_prob)

        # CNOT Layers 5-8
        c.append("CX", [cq(0), zq(0),  cq(2), zq(1),  cq(4), zq(2)])
        c.append("CX", [xq(0), cq(1),  xq(1), cq(3),  xq(2), cq(7)])
        c.append("CX", [cq(3), zq(0),  cq(5), zq(1),  cq(7), zq(2)])
        c.append("CX", [xq(0), cq(2),  xq(1), cq(4),  xq(2), cq(8)])
        c.append("H", all_xq)

    meas_targets = [xq(0), zq(0), xq(1), zq(1), xq(2), zq(2), xq(3), zq(3)]
    
    # ----------------------------------------------------
    # ROUND 0: Stable Baseline Initialization
    # ----------------------------------------------------
    apply_syndrome_cycle(circuit, inject_errors=False)
    circuit.append("M", meas_targets)
    circuit.append("R", meas_targets)
    
    # Initial deterministic Z-basis anchors
    circuit.append("DETECTOR", stim.target_rec(-7))  # zq0
    circuit.append("DETECTOR", stim.target_rec(-5))  # zq1
    circuit.append("DETECTOR", stim.target_rec(-3))  # zq2
    circuit.append("DETECTOR", stim.target_rec(-1))  # zq3

    # ----------------------------------------------------
    # ROUND 1: FAULTS TARGETED HERE
    # ----------------------------------------------------
    if data_flips:
        circuit.append("X_ERROR", [cq(i) for i in data_flips], model_prob)
        
    apply_syndrome_cycle(circuit, inject_errors=True)
    circuit.append("M", meas_targets)
    circuit.append("R", meas_targets)
    
    # Natively tracks both X and Z ancilla fluctuations relative to Round 0
    for i in range(8):
        circuit.append("DETECTOR", [stim.target_rec(-8 + i), stim.target_rec(-16 + i)])

    # ----------------------------------------------------
    # ROUND 2: Verification Buffer Round
    # ----------------------------------------------------
    apply_syndrome_cycle(circuit, inject_errors=False)
    circuit.append("M", meas_targets)
    
    # Tracks stabilization relative to Round 1
    for i in range(8):
        circuit.append("DETECTOR", [stim.target_rec(-8 + i), stim.target_rec(-16 + i)])

    # ----------------------------------------------------
    # FINAL READOUT DATA DESTRUCTION BOUNDARY
    # ----------------------------------------------------
    data_targets = [cq(i) for i in range(9)]
    circuit.append("M", data_targets)

    # Closes out the final stabilizer tracking frame (Data vs Round 2 Z-ancillas)
    circuit.append("DETECTOR", [stim.target_rec(-9), stim.target_rec(-6), stim.target_rec(-16)])  # zq0
    circuit.append("DETECTOR", [stim.target_rec(-8), stim.target_rec(-7), stim.target_rec(-5), stim.target_rec(-4), stim.target_rec(-14)])  # zq1
    circuit.append("DETECTOR", [stim.target_rec(-6), stim.target_rec(-5), stim.target_rec(-3), stim.target_rec(-2), stim.target_rec(-12)])  # zq2
    circuit.append("DETECTOR", [stim.target_rec(-4), stim.target_rec(-1), stim.target_rec(-10)])  # zq3

    circuit.append("OBSERVABLE_INCLUDE", [stim.target_rec(-9), stim.target_rec(-8), stim.target_rec(-7)], 0)

    return circuit

# --- EXECUTIVE TESTING INFRASTRUCTURE ---
if __name__ == "__main__":
    print("--- 3-Round Flat 17-Qubit Fault-Tolerant Dimensional Scan ---")
    
    # Define unified flat indices map:
    # 0 to 8: Data (9 qubits)
    # 9 to 12: Z-Ancillas (4 qubits)
    # 13 to 16: X-Ancillas (4 qubits)
    num_total_qubits = 17
    
    # Initialize the base error model graph using 1% noise across all 17 available qubits
    base_circuit = generate_3_round_fully_loaded_circuit(
        data_flips=list(range(9)), 
        z_ancilla_flips=list(range(4)), 
        x_ancilla_flips=list(range(4)), 
        model_prob=0.01
    )
    error_model = base_circuit.detector_error_model(decompose_errors=True)
    matching = pymatching.Matching.from_detector_error_model(error_model)
    
    # Loop continuously through total system fault dimensions up to weight 17
    for weight in range(1, num_total_qubits + 1):
        # Generate raw combinations out of the flat 17-qubit continuum
        combinations = list(itertools.combinations(range(num_total_qubits), weight))
        decoder_failures = 0
        
        for flat_flip_set in combinations:
            # Map the flat system indices into their target functional sub-registers
            d_flips = [i for i in flat_flip_set if 0 <= i < 9]
            z_flips = [i - 9 for i in flat_flip_set if 9 <= i < 13]
            x_flips = [i - 13 for i in flat_flip_set if 13 <= i < 17]
            
            # Build and test the unique test instance
            test_circuit = generate_3_round_fully_loaded_circuit(
                data_flips=d_flips, 
                z_ancilla_flips=z_flips, 
                x_ancilla_flips=x_flips, 
                model_prob=0.999
            )
            
            sampler = test_circuit.compile_detector_sampler()
            det_samples, obs_samples = sampler.sample(shots=1, separate_observables=True)
            
            det_flips = det_samples[0]
            actual_obs = obs_samples[0]
            
            predicted_obs = matching.decode(det_flips)
            
            if actual_obs[0] != predicted_obs[0]:
                decoder_failures += 1
                
        print(f"Dimension {weight} summary: {decoder_failures} / {len(combinations)} combinations defeated the decoder.")


--- 3-Round Flat 17-Qubit Fault-Tolerant Dimensional Scan ---
Dimension 1 summary: 0 / 17 combinations defeated the decoder.
Dimension 2 summary: 18 / 136 combinations defeated the decoder.
Dimension 3 summary: 199 / 680 combinations defeated the decoder.
Dimension 4 summary: 1008 / 2380 combinations defeated the decoder.
Dimension 5 summary: 3099 / 6188 combinations defeated the decoder.
Dimension 6 summary: 6561 / 12376 combinations defeated the decoder.
Dimension 7 summary: 10292 / 19448 combinations defeated the decoder.
Dimension 8 summary: 12442 / 24310 combinations defeated the decoder.
Dimension 9 summary: 11892 / 24310 combinations defeated the decoder.
Dimension 10 summary: 9150 / 19448 combinations defeated the decoder.
Dimension 11 summary: 5803 / 12376 combinations defeated the decoder.
Dimension 12 summary: 3082 / 6188 combinations defeated the decoder.
Dimension 13 summary: 1367 / 2380 combinations defeated the decoder.
Dimension 14 summary: 480 / 680 combinations defeat